# NETWORK / GRAPH FEATURE ENGINEERING


In [ ]:
#import libraries
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
import time
import shutil
import numpy as np
import polars as pl
import os
from pathlib import Path

Mounted at /content/drive


In [ ]:
# Set working directory to the AML project repository folder.
os.chdir(
    '/content/drive/MyDrive/AML_Project/'
    'aml-compliance-system-main'
)

# Display the current directory and its contents.
print("Current Directory:", os.getcwd())
print("Contents:", os.listdir('.'))

Current Directory: /content/drive/MyDrive/AML_Project/aml-compliance-system-main
Contents: ['docker-compose.yml', 'notebooks', 'src', 'tests', 'README.md', '.env', 'requirements.txt', 'Data', 'aml-compliance-system-main', 'gemini-svg.svg', 'str.jpeg']


In [ ]:
processed_path = "Data/processed"

print("Processed folder contents:")
print(os.listdir(processed_path))

Processed folder contents:
['accounts_processed_for_llm.csv', 'HI-Large_Cleaned.parquet', 'Behavioral_Features_Account_Level.parquet', 'transactions_features_A.csv', 'network_features', 'HI-Large_Trans_with_features.csv', 'HI-Large_Trans_Phase2_A.csv', 'chunks', 'HI-Large_Trans_Phase2_A_Final.csv', 'HI-Large_Network_Features.parquet', 'q_transformed.parquet']


## Optimized Network / Graph Feature Engineering

This version avoids rebuilding the same 179M-row lazy pipeline many times.
It creates one local checkpoint, builds the directed-pair table once, derives all
network features from that smaller table, and performs only one final join back
to the transaction dataset.


In [ ]:
cleaned_path = (
    "/content/drive/MyDrive/AML_Project/"
    "aml-compliance-system-main/"
    "Data/processed/HI-Large_Cleaned.parquet"
)

local_base_path = "/content/HI_Large_network_base.parquet"
local_pair_path = "/content/HI_Large_pair_features.parquet"
local_account_path = "/content/HI_Large_account_features.parquet"
local_final_path = "/content/HI-Large_Network_Features.parquet"

drive_final_path = (
    "/content/drive/MyDrive/AML_Project/"
    "aml-compliance-system-main/"
    "Data/processed/HI-Large_Network_Features.parquet"
)

print("Paths configured.")
print("Final Drive path:", drive_final_path)

Paths configured.
Final Drive path: /content/drive/MyDrive/AML_Project/aml-compliance-system-main/Data/processed/HI-Large_Network_Features.parquet


In [ ]:
# STEP 1 — DEDUPLICATE ONCE, CREATE NODE IDs ONCE, CHECKPOINT LOCALLY
t0 = time.time()

transaction_columns = [
    "Timestamp",
    "From Bank",
    "Account",
    "To Bank",
    "Account_duplicated_0",
    "Amount Received",
    "Receiving Currency",
    "Amount Paid",
    "Payment Currency",
    "Payment Format"
]

base = (
    pl.scan_parquet(cleaned_path)
    .unique(
        subset=transaction_columns,
        keep="first",
        maintain_order=False
    )
    .with_columns([
        pl.concat_str([
            pl.col("From Bank").cast(pl.String),
            pl.lit("_"),
            pl.col("Account").cast(pl.String)
        ]).alias("sender_id"),

        pl.concat_str([
            pl.col("To Bank").cast(pl.String),
            pl.lit("_"),
            pl.col("Account_duplicated_0").cast(pl.String)
        ]).alias("receiver_id")
    ])
)

# Materialize ONCE. Snappy is faster than zstd for this checkpoint.
base.sink_parquet(local_base_path, compression="snappy")

print(f"Base checkpoint saved in {(time.time()-t0)/60:.1f} min")
print("Local path:", local_base_path)
print("Size GB:", round(os.path.getsize(local_base_path) / (1024**3), 2))


Base checkpoint saved in 5.4 min
Local path: /content/HI_Large_network_base.parquet
Size GB: 15.07


In [ ]:
# STEP 2 — BUILD UNIQUE DIRECTED PAIRS + FREQUENCY ONCE
t0 = time.time()

base = pl.scan_parquet(local_base_path)

pair_base = (
    base
    .select(["sender_id", "receiver_id"])
    .group_by(["sender_id", "receiver_id"])
    .agg(
        pl.len().alias("sender_receiver_frequency")
    )
)

pair_base.sink_parquet(local_pair_path, compression="snappy")

print(f"Directed pair table saved in {(time.time()-t0)/60:.1f} min")
print("Local path:", local_pair_path)


Directed pair table saved in 0.2 min
Local path: /content/HI_Large_pair_features.parquet


In [ ]:
# STEP 3 — ADD BIDIRECTIONAL RELATIONSHIP TO THE SMALLER PAIR TABLE
t0 = time.time()

pairs = pl.scan_parquet(local_pair_path)

reverse_lookup = (
    pairs
    .select([
        pl.col("receiver_id").alias("sender_id"),
        pl.col("sender_id").alias("receiver_id")
    ])
    .with_columns(
        pl.lit(1, dtype=pl.Int8).alias("bidirectional_relationship")
    )
)

pair_features = (
    pairs
    .join(
        reverse_lookup,
        on=["sender_id", "receiver_id"],
        how="left"
    )
    .with_columns(
        pl.col("bidirectional_relationship")
        .fill_null(0)
        .cast(pl.Int8)
    )
)

# Overwrite through a temporary file because source and destination should not be the same scan.
pair_features_tmp = "/content/HI_Large_pair_features_complete.parquet"
pair_features.sink_parquet(pair_features_tmp, compression="snappy")
os.replace(pair_features_tmp, local_pair_path)

print(f"Bidirectional feature added in {(time.time()-t0)/60:.1f} min")


Bidirectional feature added in 0.0 min


In [ ]:
# STEP 4 — ACCOUNT-LEVEL NETWORK FEATURES FROM UNIQUE PAIRS
# Important: use the much smaller pair table, NOT the 179M transaction table.
t0 = time.time()

pairs = pl.scan_parquet(local_pair_path)

out_degree = (
    pairs
    .group_by("sender_id")
    .agg(pl.len().alias("out_degree"))
    .rename({"sender_id": "account_id"})
)

in_degree = (
    pairs
    .group_by("receiver_id")
    .agg(pl.len().alias("in_degree"))
    .rename({"receiver_id": "account_id"})
)

# Unique counterparties across both directions.
# The pair table is already unique per directed edge; .unique() removes duplicates
# where the same counterparty exists in both directions.
counterparty_edges = pl.concat([
    pairs.select([
        pl.col("sender_id").alias("account_id"),
        pl.col("receiver_id").alias("counterparty_id")
    ]),
    pairs.select([
        pl.col("receiver_id").alias("account_id"),
        pl.col("sender_id").alias("counterparty_id")
    ])
])

unique_counterparties = (
    counterparty_edges
    .unique(subset=["account_id", "counterparty_id"])
    .group_by("account_id")
    .agg(pl.len().alias("unique_counterparties"))
)

account_features = (
    out_degree
    .join(in_degree, on="account_id", how="full", coalesce=True)
    .join(unique_counterparties, on="account_id", how="left")
    .with_columns([
        pl.col("out_degree").fill_null(0),
        pl.col("in_degree").fill_null(0),
        pl.col("unique_counterparties").fill_null(0)
    ])
)

account_features.sink_parquet(local_account_path, compression="snappy")

print(f"Account feature table saved in {(time.time()-t0)/60:.1f} min")
print("Local path:", local_account_path)


Account feature table saved in 0.0 min
Local path: /content/HI_Large_account_features.parquet


In [ ]:
# STEP 5 — FINAL JOIN: TRANSACTIONS + ACCOUNT FEATURES + PAIR FEATURES
t0 = time.time()

base = pl.scan_parquet(local_base_path)
account_features = pl.scan_parquet(local_account_path)
pair_features = pl.scan_parquet(local_pair_path)

sender_features = account_features.rename({
    "account_id": "sender_id",
    "out_degree": "sender_out_degree",
    "in_degree": "sender_in_degree",
    "unique_counterparties": "sender_unique_counterparties"
})

receiver_features = account_features.rename({
    "account_id": "receiver_id",
    "out_degree": "receiver_out_degree",
    "in_degree": "receiver_in_degree",
    "unique_counterparties": "receiver_unique_counterparties"
})

df_network = (
    base
    .join(sender_features, on="sender_id", how="left")
    .join(receiver_features, on="receiver_id", how="left")
    .join(
        pair_features,
        on=["sender_id", "receiver_id"],
        how="left"
    )
)

# Save locally first so Google Drive I/O does not slow the heavy join.
df_network.sink_parquet(local_final_path, compression="snappy")

print(f"Final local network dataset created in {(time.time()-t0)/60:.1f} min")
print("Local size GB:", round(os.path.getsize(local_final_path) / (1024**3), 2))


Final local network dataset created in 2.5 min
Local size GB: 15.72


In [ ]:
# STEP 6 — VALIDATE BEFORE COPYING TO DRIVE
t0 = time.time()

final_scan = pl.scan_parquet(local_final_path)

validation = (
    final_scan
    .select([
        pl.len().alias("rows"),
        pl.col("sender_out_degree").null_count().alias("null_sender_out_degree"),
        pl.col("receiver_in_degree").null_count().alias("null_receiver_in_degree"),
        pl.col("sender_receiver_frequency").null_count().alias("null_pair_frequency"),
        pl.col("bidirectional_relationship").null_count().alias("null_bidirectional")
    ])
    .collect()
)

print(validation)
print(f"Validation finished in {(time.time()-t0)/60:.1f} min")


shape: (1, 5)
┌───────────┬─────────────────────┬─────────────────────┬─────────────────────┬────────────────────┐
│ rows      ┆ null_sender_out_deg ┆ null_receiver_in_de ┆ null_pair_frequency ┆ null_bidirectional │
│ ---       ┆ ree                 ┆ gree                ┆ ---                 ┆ ---                │
│ u32       ┆ ---                 ┆ ---                 ┆ u32                 ┆ u32                │
│           ┆ u32                 ┆ u32                 ┆                     ┆                    │
╞═══════════╪═════════════════════╪═════════════════════╪═════════════════════╪════════════════════╡
│ 179702079 ┆ 0                   ┆ 0                   ┆ 0                   ┆ 0                  │
└───────────┴─────────────────────┴─────────────────────┴─────────────────────┴────────────────────┘
Validation finished in 0.0 min


In [ ]:
# STEP 7 — COPY FINAL FILE TO GOOGLE DRIVE
t0 = time.time()

print("Local file exists:", os.path.exists(local_final_path))

if not os.path.exists(local_final_path):
    raise FileNotFoundError(
        f"Final local file was not found: {local_final_path}"
    )

os.makedirs(os.path.dirname(drive_final_path), exist_ok=True)

print("Copying file to Google Drive...")
shutil.copy2(local_final_path, drive_final_path)

print("Copy completed.")
print("Drive file exists:", os.path.exists(drive_final_path))
print("Saved at:", drive_final_path)

if os.path.exists(drive_final_path):
    print(
        "Drive file size:",
        round(os.path.getsize(drive_final_path) / (1024**3), 2),
        "GB"
    )

print(f"Time: {(time.time()-t0)/60:.1f} minutes")

Local file exists: True
Copying file to Google Drive...
Copy completed.
Drive file exists: True
Saved at: /content/drive/MyDrive/AML_Project/aml-compliance-system-main/Data/processed/HI-Large_Network_Features.parquet
Drive file size: 15.72 GB
Time: 0.8 minutes


In [ ]:
# Folder in Google Drive
drive_folder = (
    "/content/drive/MyDrive/AML_Project/"
    "aml-compliance-system-main/"
    "Data/processed/network_features"
)

os.makedirs(drive_folder, exist_ok=True)

files_to_copy = [
    "/content/HI-Large_Network_Features.parquet",
    "/content/HI_Large_account_features.parquet",
    "/content/HI_Large_network_base.parquet",
    "/content/HI_Large_pair_features.parquet"
]

for src in files_to_copy:

    if not os.path.exists(src):
        print(f"NOT FOUND: {src}")
        continue

    filename = os.path.basename(src)
    dst = os.path.join(drive_folder, filename)

    size_gb = os.path.getsize(src) / (1024**3)

    print(f"\nCopying: {filename}")
    print(f"Size: {size_gb:.2f} GB")

    start = time.time()

    shutil.copy2(src, dst)

    print(f"Saved to Drive: {dst}")
    print(f"Time: {(time.time() - start)/60:.1f} min")

print("\nDONE — files copied to Google Drive.")


Copying: HI-Large_Network_Features.parquet
Size: 15.72 GB
Saved to Drive: /content/drive/MyDrive/AML_Project/aml-compliance-system-main/Data/processed/network_features/HI-Large_Network_Features.parquet
Time: 0.8 min

Copying: HI_Large_account_features.parquet
Size: 0.03 GB
Saved to Drive: /content/drive/MyDrive/AML_Project/aml-compliance-system-main/Data/processed/network_features/HI_Large_account_features.parquet
Time: 0.1 min

Copying: HI_Large_network_base.parquet
Size: 15.07 GB
Saved to Drive: /content/drive/MyDrive/AML_Project/aml-compliance-system-main/Data/processed/network_features/HI_Large_network_base.parquet
Time: 1.3 min

Copying: HI_Large_pair_features.parquet
Size: 0.23 GB
Saved to Drive: /content/drive/MyDrive/AML_Project/aml-compliance-system-main/Data/processed/network_features/HI_Large_pair_features.parquet
Time: 0.0 min

DONE — files copied to Google Drive.
